# MMPose vs MediaPipe Pose Comparison

Run this notebook on Google Colab with a T4 GPU runtime. It writes MMPose artifacts next to the existing MediaPipe pose artifacts and produces rule/classifier comparison tables.

### 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### 2. Install MMPose

In [ ]:
# Colab Python 3.12-friendly RTMW/RTMPose runtime. Avoid mmcv/mmdet native ops.
!pip uninstall -y -q openmim openxlab chumpy mmpose mmdet mmcv mmcv-lite mmengine xtcocotools
!pip install -q -U pip wheel "setuptools>=69"
!pip install -q -U rtmlib onnxruntime-gpu opencv-python numpy tqdm

In [ ]:
import torch
import rtmlib
from rtmlib import Wholebody

print('cuda_available:', torch.cuda.is_available())
print('gpu:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')
print('torch:', torch.__version__)
print('rtmlib:', getattr(rtmlib, '__version__', 'unknown'))
print('rtmlib Wholebody import: OK')

### 3. Configure Paths

In [ ]:
from pathlib import Path
import subprocess

PROJECT_ROOT = Path('/content/drive/MyDrive/x-coach')
DATASET_ROOT = PROJECT_ROOT / 'data' / 'Fitness-AQA_dataset_release' / 'Squat' / 'Labeled_Dataset'
OUTPUT_ROOT = PROJECT_ROOT / 'data' / 'Squat' / 'Labeled_Dataset'
SPLIT_DIR = DATASET_ROOT / 'Splits'
LOCAL_VIDEO_DIR = Path('/content/local_videos')
LOCAL_VIDEO_DIR.mkdir(parents=True, exist_ok=True)

video_zip = DATASET_ROOT / 'videos.zip'
if video_zip.exists():
    subprocess.run(['unzip', '-q', '-n', str(video_zip), '-d', str(LOCAL_VIDEO_DIR)], check=True)

VIDEO_ROOT = LOCAL_VIDEO_DIR if any(LOCAL_VIDEO_DIR.rglob('*.mp4')) else DATASET_ROOT / 'videos'
MMPOSE_JSON_DIR = OUTPUT_ROOT / 'mmpose_pose_json'
MMPOSE_FEATURE_DIR = OUTPUT_ROOT / 'mmpose_pose_features'
MMPOSE_VIEW_METADATA = OUTPUT_ROOT / 'mmpose_view_metadata.csv'
MMPOSE_RULE_DIR = OUTPUT_ROOT / 'mmpose_pose_rule_detections'
MMPOSE_RULE_SUMMARY = OUTPUT_ROOT / 'mmpose_pose_rule_detections_summary.csv'
MMPOSE_RULE_METRICS = OUTPUT_ROOT / 'mmpose_pose_rule_validation_metrics.csv'
MMPOSE_CLASSIFIER_ROOT = PROJECT_ROOT / 'data' / 'Squat' / 'mmpose_pose_classifier_experiments'

print('PROJECT_ROOT:', PROJECT_ROOT)
print('DATASET_ROOT:', DATASET_ROOT)
print('OUTPUT_ROOT:', OUTPUT_ROOT)
print('VIDEO_ROOT:', VIDEO_ROOT)

### 4. Extract MMPose Whole-Body JSON

In [ ]:
cmd = [
    'python', 'scripts/run_mmpose_pose_extraction.py',
    '--video-dir', str(VIDEO_ROOT),
    '--split-dir', str(SPLIT_DIR),
    '--output-dir', str(MMPOSE_JSON_DIR),
    '--runtime', 'rtmlib',
    '--model', 'balanced',
    '--device', 'cuda:0',
]
subprocess.run(cmd, cwd=PROJECT_ROOT, check=True)

### 5. Convert MMPose JSON to Pose Features

In [ ]:
cmd = [
    'python', 'scripts/run_pose_feature_extraction.py',
    '--pose-json-dir', str(MMPOSE_JSON_DIR),
    '--split-dir', str(SPLIT_DIR),
    '--output-dir', str(MMPOSE_FEATURE_DIR),
    '--overwrite',
]
subprocess.run(cmd, cwd=PROJECT_ROOT, check=True)

### 6. View Metadata and Rule Evaluation

In [ ]:
subprocess.run([
    'python', 'scripts/run_view_estimation.py',
    '--pose-json-dir', str(MMPOSE_JSON_DIR),
    '--split-dir', str(SPLIT_DIR),
    '--output', str(MMPOSE_VIEW_METADATA),
], cwd=PROJECT_ROOT, check=True)

subprocess.run([
    'python', 'scripts/run_pose_rule_detection.py',
    '--pose-json-dir', str(MMPOSE_JSON_DIR),
    '--split-dir', str(SPLIT_DIR),
    '--output-dir', str(MMPOSE_RULE_DIR),
    '--summary-output', str(MMPOSE_RULE_SUMMARY),
    '--no-retrieval',
], cwd=PROJECT_ROOT, check=True)

subprocess.run([
    'python', 'scripts/evaluate_pose_rule_detection.py',
    '--detections-dir', str(MMPOSE_RULE_DIR),
    '--view-metadata', str(MMPOSE_VIEW_METADATA),
    '--output', str(MMPOSE_RULE_METRICS),
], cwd=PROJECT_ROOT, check=True)

### 7. Train MMPose Pose-Only Classifiers

In [ ]:
subprocess.run([
    'python', 'scripts/run_videomae_experiment_grid.py',
    '--feature-dir', str(MMPOSE_FEATURE_DIR),
    '--train-keys', str(SPLIT_DIR / 'train_keys.json'),
    '--val-keys', str(SPLIT_DIR / 'val_keys.json'),
    '--test-keys', str(SPLIT_DIR / 'test_keys.json'),
    '--forward-labels', str(DATASET_ROOT / 'Labels' / 'error_knees_forward.json'),
    '--inward-labels', str(DATASET_ROOT / 'Labels' / 'error_knees_inward.json'),
    '--output-root', str(MMPOSE_CLASSIFIER_ROOT),
    '--label-modes', 'combined,knees_forward,knees_inward',
    '--seeds', '1,2,3,4,5',
    '--epochs', '20',
    '--lr', '3e-4',
    '--hidden-dim', '128',
    '--dropout', '0.4',
    '--weight-decay', '0.01',
    '--early-stopping-patience', '5',
    '--threshold-objective', 'balanced_accuracy',
    '--device', 'cuda',
    '--normalize-features',
], cwd=PROJECT_ROOT, check=True)

### 8. Write Backend Comparison Report

In [ ]:
subprocess.run([
    'python', 'scripts/compare_pose_backends.py',
    '--mmpose-pose-json-dir', str(MMPOSE_JSON_DIR),
    '--mmpose-rule-metrics', str(MMPOSE_RULE_METRICS),
    '--mmpose-classifier-summary', str(MMPOSE_CLASSIFIER_ROOT / 'metrics' / 'experiment_summary.csv'),
], cwd=PROJECT_ROOT, check=True)

comparison_md = PROJECT_ROOT / 'data' / 'Squat' / 'mmpose_mediapipe_comparison' / 'backend_comparison.md'
print(comparison_md.read_text()[:4000])